In [6]:
! pip install --upgrade pyspark

## Imported the required libraries

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, col, lit, sum, count, desc, when, row_number
from pyspark.sql.window import Window

## Creating a session

In [8]:
spark = SparkSession.builder.appName("PySpark_Basics").getOrCreate()
print(f"Spark Session Created Successfully with Spark version: {spark.version}")

Exception in thread "main" java.lang.UnsupportedClassVersionError: org/apache/spark/launcher/Main has been compiled by a more recent version of the Java Runtime (class file version 61.0), this version of the Java Runtime only recognizes class file versions up to 52.0
	at java.lang.ClassLoader.defineClass1(Native Method)
	at java.lang.ClassLoader.defineClass(ClassLoader.java:756)
	at java.security.SecureClassLoader.defineClass(SecureClassLoader.java:142)
	at java.net.URLClassLoader.defineClass(URLClassLoader.java:473)
	at java.net.URLClassLoader.access$100(URLClassLoader.java:74)
	at java.net.URLClassLoader$1.run(URLClassLoader.java:369)
	at java.net.URLClassLoader$1.run(URLClassLoader.java:363)
	at java.security.AccessController.doPrivileged(Native Method)
	at java.net.URLClassLoader.findClass(URLClassLoader.java:362)
	at java.lang.ClassLoader.loadClass(ClassLoader.java:418)
	at sun.misc.Launcher$AppClassLoader.loadClass(Launcher.java:371)
	at java.lang.ClassLoader.loadClass(ClassLoade

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

## Create Random Data as DataFrame

In [ ]:
employee_data = [
    (101, "Gourav Yadav", "IT", 100000, "New Delhi"),
    (102, "Dev Yadav", "IT", 80000, "Mumbai"),
    (103, "Gourav Kumar", "Sales", 70000, "Chennai"),
    (104, "Amit Yadav", "IT", 100000, "New Delhi"),
    (105, "Divya Yadav", "HR", 50000, "Bangalore"),
    (106, "Ajay verma", "IT", 90000, "Ahmedabad"),
    (107, "Saurabh Raj", "IT", 10000, "Pune"),
]
column = ["emp_id", "emp_name","dept", "salary", "city"]

df = spark.createDataFrame(
    data = employee_data,
    schema = column
)

NameError: name 'spark' is not defined

In [ ]:
print("Original Schema\n")
df.show()
df.printSchema()

Original Schema

+------+------------+-----+------+---------+
|emp_id|    emp_name| dept|salary|     city|
+------+------------+-----+------+---------+
|   101|Gourav Yadav|   IT|100000|New Delhi|
|   102|   Dev Yadav|   IT| 80000|   Mumbai|
|   103|Gourav Kumar|Sales| 70000|  Chennai|
|   104|  Amit Yadav|   IT|100000|New Delhi|
|   105| Divya Yadav|   HR| 50000|Bangalore|
|   106|  Ajay verma|   IT| 90000|Ahmedabad|
|   107| Saurabh Raj|   IT| 10000|     Pune|
+------+------------+-----+------+---------+

root
 |-- emp_id: long (nullable = true)
 |-- emp_name: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- salary: long (nullable = true)
 |-- city: string (nullable = true)



## Data Transformation

### 1. Selecting and Renaming columns

In [ ]:
df_selected = df.select(col('emp_id'), col('emp_name').alias("full_name"), col('salary'))
df_selected.show(5)

+------+------------+------+
|emp_id|   full_name|salary|
+------+------------+------+
|   101|Gourav Yadav|100000|
|   102|   Dev Yadav| 80000|
|   103|Gourav Kumar| 70000|
|   104|  Amit Yadav|100000|
|   105| Divya Yadav| 50000|
+------+------------+------+
only showing top 5 rows


### 2. Filtering Rows

In [ ]:
df_filtered = df.filter((col("dept") == "IT") & (col("salary") > 80000))
df_filtered.show()

+------+------------+----+------+---------+
|emp_id|    emp_name|dept|salary|     city|
+------+------------+----+------+---------+
|   101|Gourav Yadav|  IT|100000|New Delhi|
|   104|  Amit Yadav|  IT|100000|New Delhi|
|   106|  Ajay verma|  IT| 90000|Ahmedabad|
+------+------------+----+------+---------+



### 3. Adding and Modifying Columns

In [ ]:
df_transformed = df.withColumn("bonus", col("salary") * 0.10).withColumn("salary_tier",
                                                                         when(col("salary") >= 90000,
                                                                        lit("High")).otherwise(lit("Standard")))
df_transformed.show()

+------+------------+-----+------+---------+-------+-----------+
|emp_id|    emp_name| dept|salary|     city|  bonus|salary_tier|
+------+------------+-----+------+---------+-------+-----------+
|   101|Gourav Yadav|   IT|100000|New Delhi|10000.0|       High|
|   102|   Dev Yadav|   IT| 80000|   Mumbai| 8000.0|   Standard|
|   103|Gourav Kumar|Sales| 70000|  Chennai| 7000.0|   Standard|
|   104|  Amit Yadav|   IT|100000|New Delhi|10000.0|       High|
|   105| Divya Yadav|   HR| 50000|Bangalore| 5000.0|   Standard|
|   106|  Ajay verma|   IT| 90000|Ahmedabad| 9000.0|       High|
|   107| Saurabh Raj|   IT| 10000|     Pune| 1000.0|   Standard|
+------+------------+-----+------+---------+-------+-----------+



In [ ]:
df_transformed.explain(True)

== Parsed Logical Plan ==
'Project [unresolvedstarwithcolumns(salary_tier, CASE WHEN '`>=`('salary, 90000) THEN High ELSE Standard END, None)]
+- Project [emp_id#261L, emp_name#262, dept#263, salary#264L, city#265, (cast(salary#264L as double) * 0.1) AS bonus#309]
   +- LogicalRDD [emp_id#261L, emp_name#262, dept#263, salary#264L, city#265], false

== Analyzed Logical Plan ==
emp_id: bigint, emp_name: string, dept: string, salary: bigint, city: string, bonus: double, salary_tier: string
Project [emp_id#261L, emp_name#262, dept#263, salary#264L, city#265, bonus#309, CASE WHEN (salary#264L >= cast(90000 as bigint)) THEN High ELSE Standard END AS salary_tier#310]
+- Project [emp_id#261L, emp_name#262, dept#263, salary#264L, city#265, (cast(salary#264L as double) * 0.1) AS bonus#309]
   +- LogicalRDD [emp_id#261L, emp_name#262, dept#263, salary#264L, city#265], false

== Optimized Logical Plan ==
Project [emp_id#261L, emp_name#262, dept#263, salary#264L, city#265, (cast(salary#264L as doub

### 4. Aggregation & Grouping

In [ ]:
df_agg = df.groupby("dept").agg(avg("salary").alias("avg_salary"),
                                count("emp_id").alias("emp_count")).orderBy(desc("avg_salary"))
df_agg.show()

+-----+----------+---------+
| dept|avg_salary|emp_count|
+-----+----------+---------+
|   IT|   76000.0|        5|
|Sales|   70000.0|        1|
|   HR|   50000.0|        1|
+-----+----------+---------+



### 5. Window Function

In [ ]:
window_spec = Window.partitionBy("dept").orderBy(desc("salary"))
df_ranked = df.withColumn("salary_rank", row_number().over(window_spec))
df_ranked.show()

+------+------------+-----+------+---------+-----------+
|emp_id|    emp_name| dept|salary|     city|salary_rank|
+------+------------+-----+------+---------+-----------+
|   105| Divya Yadav|   HR| 50000|Bangalore|          1|
|   101|Gourav Yadav|   IT|100000|New Delhi|          1|
|   104|  Amit Yadav|   IT|100000|New Delhi|          2|
|   106|  Ajay verma|   IT| 90000|Ahmedabad|          3|
|   102|   Dev Yadav|   IT| 80000|   Mumbai|          4|
|   107| Saurabh Raj|   IT| 10000|     Pune|          5|
|   103|Gourav Kumar|Sales| 70000|  Chennai|          1|
+------+------------+-----+------+---------+-----------+



## Stopping the Spark Session

In [ ]:
spark.stop()
print("\n Spark Session Stopped Successfully")


 Spark Session Stopped Successfully


# Directed Acyclic Graph (DAG)

## Step 1. Initialization and Setup

In [ ]:
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit

Session Creation

In [ ]:
spark = SparkSession.builder.appName("DAG session").getOrCreate()
print("Spark Session has been created successfully")

Spark Session has been created successfully


Lazy Evaluation

### Create a large range of numbers (10 millio rows)

In [ ]:
start_time = time.time()
df_range = spark.range(0,10000000)
print(f"Time to define range: {time.time() - start_time: .4f} seconds (Almost Instantaneous)")

Time to define range:  0.0090 seconds (Almost Instantaneous)


### Add an expensive mathematical transformations

In [ ]:
df_transformed = df_range.withColumn("multiplied_val",
                                     col("id")*3).filter(col("multiplied_val")%2 == 0)
print(f"Time to define transformations: {time.time() - start_time:.4f} seconds (Still no work is done)")

Time to define transformations: 2.1383 seconds (Still no work is done)


## Viewing the Logical and Physical DAG

In [ ]:
df_transformed.explain(True)

== Parsed Logical Plan ==
'Filter '`=`('`%`('multiplied_val, 2), 0)
+- Project [id#384L, (id#384L * cast(3 as bigint)) AS multiplied_val#386L]
   +- Range (0, 10000000, step=1, splits=Some(2))

== Analyzed Logical Plan ==
id: bigint, multiplied_val: bigint
Filter ((multiplied_val#386L % cast(2 as bigint)) = cast(0 as bigint))
+- Project [id#384L, (id#384L * cast(3 as bigint)) AS multiplied_val#386L]
   +- Range (0, 10000000, step=1, splits=Some(2))

== Optimized Logical Plan ==
Project [id#384L, (id#384L * 3) AS multiplied_val#386L]
+- Filter (((id#384L * 3) % 2) = 0)
   +- Range (0, 10000000, step=1, splits=Some(2))

== Physical Plan ==
*(1) Project [id#384L, (id#384L * 3) AS multiplied_val#386L]
+- *(1) Filter (((id#384L * 3) % 2) = 0)
   +- *(1) Range (0, 10000000, step=1, splits=2)



## Trigger the Action

In [ ]:
action_start_time = time.time()
total_rows = df_transformed.count()
action_end_time = time.time()

print(f"Resulting Row Count: {total_rows}")
print(f"Time taken to execute Action: {action_end_time - action_start_time: .2f} seconds")


Resulting Row Count: 5000000
Time taken to execute Action:  0.19 seconds


## Performance Optimization

In [ ]:
## Re-run the whole action without cache
start_time = time.time()
df_transformed.count()
print(f"Time for 2nd run (No Cache): {time.time() - start_time: .2f} seconds")

Time for 2nd run (No Cache):  0.63 seconds


In [ ]:
## Re-run with Cache
## .cache() is a lazy transformation that tells Spark to store the result in memory once computed

df_transformed.cache()
df_transformed.count()
start_time = time.time()
df_transformed.count()
print(f"Time for 3rd run (Cached): {time.time() - start_time: .2f} seconds")

Time for 3rd run (Cached):  0.25 seconds
